This is the start of a new project

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt, find_peaks

In [9]:
DATA_DIR = Path(r"src")
CSV_FILE = DATA_DIR / "Running trial 2.csv"

CSV_FILE = DATA_DIR / "Running trial 2.csv"
mb = 59.8
g = 9.81
BW = mb * g
m1 = 0.08 * mb
m2 = 0.92 * mb

In [10]:
print("CSV file:", CSV_FILE)
print("Body mass =", mb)
print("Body weight =", BW)
print("m1 =", m1)
print("m2 =", m2)

CSV file: src\Running trial 2.csv
Body mass = 59.8
Body weight = 586.638
m1 = 4.784
m2 = 55.016


In [11]:
df = pd.read_csv(CSV_FILE)

print("Columns:")
for c in df.columns:
    print(c)

time = df["Time"].to_numpy()
fs = round(1 / np.mean(np.diff(time)))
dt = 1 / fs

print("\nSampling frequency =", fs)


Columns:
Frame
Time
mal_lat_left_X
mal_lat_left_Y
mal_lat_left_Z
calc_back_left_X
calc_back_left_Y
calc_back_left_Z
toe_left_X
toe_left_Y
toe_left_Z
mal_lat_right_X
mal_lat_right_Y
mal_lat_right_Z
calc_back_right_X
calc_back_right_Y
calc_back_right_Z
toe_right_X
toe_right_Y
toe_right_Z
C7_X
C7_Y
C7_Z

Sampling frequency = 200


In [12]:
heelL = df["calc_back_left_Z"].to_numpy()
toeL  = df["toe_left_Z"].to_numpy()
ankL  = df["mal_lat_left_Z"].to_numpy() / 1000

heelR = df["calc_back_right_Z"].to_numpy()
toeR  = df["toe_right_Z"].to_numpy()
ankR  = df["mal_lat_right_Z"].to_numpy() / 1000

fc = 25
b, a = butter(4, fc/(fs/2), btype="low")

heelZ_L = filtfilt(b, a, heelL)
toeZ_L  = filtfilt(b, a, toeL)
ankZ_L  = filtfilt(b, a, ankL)

heelZ_R = filtfilt(b, a, heelR)
toeZ_R  = filtfilt(b, a, toeR)
ankZ_R  = filtfilt(b, a, ankR)

print("Filtering done")

Filtering done


In [13]:
locs_L, _ = find_peaks(-heelZ_L, distance=round(0.30 * fs))
locs_R, _ = find_peaks(-heelZ_R, distance=round(0.30 * fs))

print("Left touchdown candidates =", len(locs_L))
print("Right touchdown candidates =", len(locs_R))
print("Total touchdown candidates =", len(locs_L) + len(locs_R))

Left touchdown candidates = 44
Right touchdown candidates = 45
Total touchdown candidates = 89


In [14]:
toeV_L = np.gradient(toeZ_L, dt)
toeV_R = np.gradient(toeZ_R, dt)

print("Toe velocity computed")


Toe velocity computed


In [15]:
def pair_full_steps(locs, toeV, time, fs):
    TD = []
    TO = []
    nextTD = []
    tc = []
    ta = []

    for i in range(len(locs) - 1):
        td = locs[i]
        td_next = locs[i + 1]

        search_start = td + round(0.12 * fs)
        search_end = min(td + round(0.35 * fs), td_next - 1)

        if search_end <= search_start:
            continue

        seg = toeV[search_start:search_end + 1]
        idx_max = np.argmax(seg)
        to = search_start + idx_max

        TD.append(time[td])
        TO.append(time[to])
        nextTD.append(time[td_next])
        tc.append((to - td) / fs)
        ta.append((td_next - to) / fs)

    return (
        np.array(TD),
        np.array(TO),
        np.array(nextTD),
        np.array(tc),
        np.array(ta),
    )

TD_L_full, TO_L_full, nextTD_L_full, tc_L_full, ta_L_full = pair_full_steps(locs_L, toeV_L, time, fs)
TD_R_full, TO_R_full, nextTD_R_full, tc_R_full, ta_R_full = pair_full_steps(locs_R, toeV_R, time, fs)

print("Left paired steps =", len(tc_L_full))
print("Right paired steps =", len(tc_R_full))
print("Total paired steps =", len(tc_L_full) + len(tc_R_full))

Left paired steps = 43
Right paired steps = 44
Total paired steps = 87
